In [3]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface

# 用于PSTH计算
import neo
from quantities import ms
from elephant.statistics import instantaneous_rate
from elephant.kernels import GaussianKernel
import pickle


In [4]:
file_list = os.listdir(f"/media/ubuntu/sda/mouse_test/raw_data/WLF_128chmouse1_natima_RHD_251129_183351")
file_list.remove("settings.xml")
recording_raw_list = []
file_list = sorted(file_list)

for file in file_list:
    recording_raw_list.append(se.read_intan(f"/media/ubuntu/sda/mouse_test/raw_data/WLF_128chmouse1_natima_RHD_251129_183351/{file}", stream_id= '4'))
recording_raw = concatenate_recordings(recording_list=recording_raw_list)

# recording_raw = spre.unsigned_to_signed(recording_raw)
# recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
# recording_recorded = spre.notch_filter(recording_recorded, freq=50)
# recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [5]:
trigger = recording_raw.get_traces().astype(int).flatten()


In [6]:
trigger_indices = np.where(trigger > 0.5)[0]

In [7]:
start_indices = trigger_indices[np.concatenate(([True], np.diff(trigger_indices) > 1))]

print(f"连续段起点的数量: {len(start_indices)}")
print(f"前10个起点: {start_indices[:10]}")
print(f"第一个起点: {start_indices[0]}")  # 应该就是6334


连续段起点的数量: 2374
前10个起点: [175015 185358 195678 206016 216427 226803 237138 247493 257913 268273]
第一个起点: 175015


In [8]:
trigger_log = pd.read_csv("/media/ubuntu/sda/mouse_test/trigger/WLF_128chmouse1_natima_RHD_251129_183351.csv")

In [9]:
# 在trigger_log上添加一列来映射start_indices
# 规则：非rest行对应1个start_indices，rest行对应2个start_indices
trigger_log['start_index'] = None
trigger_log['start_index_2'] = None  # 用于rest行的第二个trigger

start_idx_counter = 0  # 用于追踪当前使用的start_indices位置

for i in range(len(trigger_log)):
    if trigger_log.loc[i, 'info_type'] == 'rest':
        # rest行对应两个start_indices
        if start_idx_counter < len(start_indices):
            trigger_log.loc[i, 'start_index'] = start_indices[start_idx_counter]
            start_idx_counter += 1
        if start_idx_counter < len(start_indices):
            trigger_log.loc[i, 'start_index_2'] = start_indices[start_idx_counter]
            start_idx_counter += 1
    else:
        # 非rest行对应一个start_indices
        if start_idx_counter < len(start_indices):
            trigger_log.loc[i, 'start_index'] = start_indices[start_idx_counter]
            start_idx_counter += 1

print(f"已映射的start_indices数量: {start_idx_counter}")
print(f"start_indices总数: {len(start_indices)}")
print(f"trigger_log行数: {len(trigger_log)}")

已映射的start_indices数量: 2374
start_indices总数: 2374
trigger_log行数: 2367


In [10]:
trigger_log = trigger_log[trigger_log['info_type'] != 'rest']

In [11]:
trigger_log['image'] = None
trigger_log.index = range(len(trigger_log))
for i in range(len(trigger_log)):
    trigger_log.loc[i, 'image'] = trigger_log.loc[i, 'image_name'].split('.')[0].split("_")[2]

In [12]:
trigger_log['start_extended'] = trigger_log['start_index'] - 0.1 * 20000
trigger_log['end_index'] = trigger_log['start_index'] + 0.25 * 20000
trigger_log['end_extended'] = trigger_log['start_index'] + 0.35 * 20000


In [13]:
trigger_log.to_csv("/media/ubuntu/sda/mouse_test/trigger/trigger_inf_WLF_128chmouse1_natima_RHD_251129_183351.csv")

In [14]:
spike_inf = pd.read_csv("/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351/spike_inf.tsv", sep = '\t')
import pickle
with open("/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351/neuron_inf.pkl", 'rb') as f:
    neuron_inf = pickle.load(f)

In [15]:
# 获取采样频率和基本参数
sampling_rate = 20000  # Hz
extend_time = 0.1  # 秒，左右延长0.1s
stimulus_duration = 0.25  # 秒，刺激持续时间

# 确保neuron列是字符串类型，并过滤NaN值
spike_inf['neuron'] = spike_inf['neuron'].astype(str)
spike_inf = spike_inf[spike_inf['neuron'] != 'nan']  # 移除NaN值
spike_inf = spike_inf[spike_inf['neuron'] != 'None']  # 移除None值

# 确保image列是字符串类型，并过滤NaN值
trigger_log['image'] = trigger_log['image'].astype(str)
trigger_log = trigger_log[trigger_log['image'] != 'nan']  # 移除NaN值
trigger_log = trigger_log[trigger_log['image'] != 'None']  # 移除None值

# 确保time列是数值类型
spike_inf['time'] = pd.to_numeric(spike_inf['time'], errors='coerce')
spike_inf = spike_inf.dropna(subset=['time'])  # 移除time为NaN的行

# 确保trigger_log中的时间相关列是数值类型
for col in ['start_index', 'end_index', 'start_extended', 'end_extended']:
    if col in trigger_log.columns:
        trigger_log[col] = pd.to_numeric(trigger_log[col], errors='coerce')

# 获取所有neuron和image（使用自然排序）
unique_neurons = sorted([str(n) for n in spike_inf['neuron'].unique() if pd.notna(n) and str(n) not in ['nan', 'None']])
unique_images = sorted([str(img) for img in trigger_log['image'].unique() if pd.notna(img) and str(img) not in ['nan', 'None']], 
                       key=lambda x: int(x) if x.isdigit() else float('inf'))
n_neurons = len(unique_neurons)
n_images = len(unique_images)

print(f"Neurons: {n_neurons}")
print(f"Images: {n_images}")
print(f"Sampling rate: {sampling_rate} Hz")
print(f"First few neurons: {unique_neurons[:5]}")
print(f"First few images: {unique_images[:5]}")


Neurons: 53
Images: 118
Sampling rate: 20000 Hz
First few neurons: ['Neuron_141', 'Neuron_147', 'Neuron_151', 'Neuron_160', 'Neuron_161']
First few images: ['0', '1', '2', '3', '4']


In [16]:
# 1. 生成Raster Plot：一个PDF，每个neuron一页，每页显示该neuron所有trials
output_dir = "/media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351"
os.makedirs(output_dir, exist_ok=True)

print("Generating raster plots...")
pdf_path = os.path.join(output_dir, "raster_all_neurons.pdf")

with PdfPages(pdf_path) as pdf:
    for neuron in unique_neurons:
        neuron_spikes = spike_inf[spike_inf['neuron'] == neuron]['time'].values
        
        # 获取所有trials（按顺序，不按image分组）
        all_trials = trigger_log.copy().reset_index(drop=True)
        n_trials = len(all_trials)
        
        if n_trials == 0:
            continue
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        for trial_idx, (_, trial) in enumerate(all_trials.iterrows()):
            start_ext = int(trial['start_extended'])
            end_ext = int(trial['end_extended'])
            
            # 获取该trial内的spikes（使用extended时间窗）
            trial_spikes = neuron_spikes[(neuron_spikes >= start_ext) & (neuron_spikes <= end_ext)]
            
            if len(trial_spikes) > 0:
                # 转换为相对时间（秒）
                relative_spikes = (trial_spikes - start_ext) / sampling_rate
                # 绘制raster
                ax.vlines(relative_spikes, trial_idx - 0.4, trial_idx + 0.4, colors='black', linewidths=1)
        
        # 标记刺激开始和结束时间
        stim_start = extend_time
        stim_end = extend_time + stimulus_duration
        ax.axvline(x=stim_start, color='red', linestyle='--', linewidth=1.5, label='Stimulus Start', alpha=0.7)
        ax.axvline(x=stim_end, color='red', linestyle='--', linewidth=1.5, label='Stimulus End', alpha=0.7)
        
        ax.set_xlabel('Time (s)', fontsize=12)
        ax.set_ylabel('Trial', fontsize=12)
        ax.set_title(f'Raster Plot: {neuron} (Total {n_trials} trials)', fontsize=14)
        ax.set_xlim(0, extend_time + stimulus_duration + extend_time)
        ax.set_ylim(-0.5, n_trials - 0.5)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        

print(f"Raster plots completed! Saved to {pdf_path}")


Generating raster plots...
Raster plots completed! Saved to /media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351/raster_all_neurons.pdf


In [17]:
# 2. 生成PSTH Lineplot：1个PDF，n_images页，每页n_neuron个子图
print("Generating PSTH lineplots...")

# 设置Gaussian kernel用于PSTH计算
gk = GaussianKernel(25 * ms)  # 25ms的Gaussian kernel
bin_size_ms = 10  # 10ms的bin size
bin_size_s = bin_size_ms / 1000.0

# 计算时间轴（使用extended时间窗）
total_time_extended = 0.45  # 0.45秒
time_bins = np.arange(0, total_time_extended, bin_size_s)
n_time_bins = len(time_bins)

pdf_path = os.path.join(output_dir, "psth_lineplot.pdf")

with PdfPages(pdf_path) as pdf:
    for image in unique_images:
        # 获取该image的所有trials
        image_trials = trigger_log[trigger_log['image'] == image]
        
        if len(image_trials) == 0:
            continue
        
        # 计算子图布局
        n_cols = int(np.ceil(np.sqrt(n_neurons)))
        n_rows = int(np.ceil(n_neurons / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 2.5))
        if n_neurons == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        fig.suptitle(f'PSTH Lineplot - Image {image}', fontsize=16, y=0.995)
        
        for neuron_idx, neuron in enumerate(unique_neurons):
            ax = axes[neuron_idx]
            neuron_spikes = spike_inf[spike_inf['neuron'] == neuron]['time'].values
            
            # 收集所有trials的PSTH
            all_psth_trials = []
            
            for _, trial in image_trials.iterrows():
                start_ext = int(trial['start_extended'])
                end_ext = int(trial['end_extended'])
                
                # 获取该trial内的spikes
                trial_spikes = neuron_spikes[(neuron_spikes >= start_ext) & (neuron_spikes <= end_ext)]
                
                if len(trial_spikes) > 0:
                    # 转换为相对时间（秒）
                    relative_spikes = (trial_spikes - start_ext) / sampling_rate
                    # 创建SpikeTrain对象
                    spiketrain = neo.SpikeTrain(relative_spikes * 1000 * ms, t_stop=total_time_extended * 1000 * ms, t_start=0 * ms)
                    # 计算instantaneous rate
                    inst_rate = instantaneous_rate(spiketrain, kernel=gk, sampling_period=bin_size_ms * ms)
                    psth_trial = inst_rate.magnitude.flatten()
                else:
                    psth_trial = np.zeros(n_time_bins)
                
                # 确保长度一致
                if len(psth_trial) < n_time_bins:
                    psth_trial = np.pad(psth_trial, (0, n_time_bins - len(psth_trial)), 'constant')
                elif len(psth_trial) > n_time_bins:
                    psth_trial = psth_trial[:n_time_bins]
                
                all_psth_trials.append(psth_trial)
            
            if len(all_psth_trials) > 0:
                all_psth_trials = np.array(all_psth_trials)  # (n_trials, n_time_bins)
                
                # 计算mean和std
                mean_psth = np.mean(all_psth_trials, axis=0)
                std_psth = np.std(all_psth_trials, axis=0)
                
                # 绘制mean lineplot
                ax.plot(time_bins, mean_psth, color='black', linewidth=2, label='Mean')
                
                # 绘制std阴影
                ax.fill_between(time_bins, mean_psth - std_psth, mean_psth + std_psth, 
                               alpha=0.3, color='gray', label='±1 SD')
                
                # 标记刺激开始和结束时间
                stim_start = extend_time
                stim_end = extend_time + stimulus_duration
                ax.axvline(x=stim_start, color='red', linestyle='--', linewidth=1, alpha=0.7)
                ax.axvline(x=stim_end, color='red', linestyle='--', linewidth=1, alpha=0.7)
            
            ax.set_title(f'{neuron}', fontsize=10)
            ax.set_xlabel('Time (s)', fontsize=9)
            ax.set_ylabel('Firing Rate (Hz)', fontsize=9)
            ax.set_xlim(0, total_time_extended)
            ax.grid(True, alpha=0.3)
            if neuron_idx == 0:
                ax.legend(fontsize=8)
        # 隐藏多余的子图
        for idx in range(n_neurons, len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        
        print(f"Saved PSTH page for image {image}")

print("PSTH lineplots completed!")


Generating PSTH lineplots...
Saved PSTH page for image 0
Saved PSTH page for image 1
Saved PSTH page for image 2
Saved PSTH page for image 3
Saved PSTH page for image 4
Saved PSTH page for image 5
Saved PSTH page for image 6
Saved PSTH page for image 7
Saved PSTH page for image 8
Saved PSTH page for image 9
Saved PSTH page for image 10
Saved PSTH page for image 11
Saved PSTH page for image 12
Saved PSTH page for image 13
Saved PSTH page for image 14
Saved PSTH page for image 15
Saved PSTH page for image 16
Saved PSTH page for image 17
Saved PSTH page for image 18
Saved PSTH page for image 19
Saved PSTH page for image 20
Saved PSTH page for image 21
Saved PSTH page for image 22
Saved PSTH page for image 23
Saved PSTH page for image 24
Saved PSTH page for image 25
Saved PSTH page for image 26
Saved PSTH page for image 27
Saved PSTH page for image 28
Saved PSTH page for image 29
Saved PSTH page for image 30
Saved PSTH page for image 31
Saved PSTH page for image 32
Saved PSTH page for imag

In [18]:
# 5. 计算反应矩阵：n_neuron * n_image * n_trial * time_windows
# 使用index时间窗（start_index到end_index）
print("Computing response matrix...")

# 计算时间bins（使用index时间窗，即刺激实际时间）
total_time_index = stimulus_duration  # 0.25秒
time_bins_index = np.arange(0, total_time_index, bin_size_s)
n_time_bins_index = len(time_bins_index)

# 确定每个image的最大trial数量
max_trials_per_image = {}
for image in unique_images:
    image_trials = trigger_log[trigger_log['image'] == image]
    max_trials_per_image[image] = len(image_trials)

max_trials = max(max_trials_per_image.values()) if max_trials_per_image else 0

print(f"Matrix dimensions:")
print(f"  - Neurons: {n_neurons}")
print(f"  - Images: {n_images}")
print(f"  - Max trials per image: {max_trials}")
print(f"  - Time bins: {n_time_bins_index}")

# 反应矩阵：n_neuron * n_image * n_trial * time_windows
response_matrix = np.full((n_neurons, n_images, max_trials, n_time_bins_index), np.nan)

# 为每个neuron、每个image、每个trial计算反应
for neuron_idx, neuron in enumerate(unique_neurons):
    neuron_spikes = spike_inf[spike_inf['neuron'] == neuron]['time'].values
    
    for image_idx, image in enumerate(unique_images):
        image_trials = trigger_log[trigger_log['image'] == image].reset_index(drop=True)
        
        for trial_idx, (_, trial) in enumerate(image_trials.iterrows()):
            if trial_idx >= max_trials:
                break
            
            start_idx = int(trial['start_index'])
            end_idx = int(trial['end_index'])
            
            # 检查时间窗口有效性
            if end_idx <= start_idx:
                continue
            
            # 计算实际时间窗口长度（秒）
            actual_time_window = 250
            
            # 获取该trial内的spikes（使用index时间窗）
            trial_spikes = neuron_spikes[(neuron_spikes >= start_idx) & (neuron_spikes < end_idx)]
            
            if len(trial_spikes) > 0:
                # 转换为相对时间（秒）
                relative_spikes = (trial_spikes - start_idx) / sampling_rate
                # 确保spikes在有效范围内
                relative_spikes = relative_spikes[relative_spikes >= 0]
                relative_spikes = relative_spikes[relative_spikes < actual_time_window]
                
                if len(relative_spikes) > 0:
                    try:
                        # 创建SpikeTrain对象，使用实际时间窗口
                        spiketrain = neo.SpikeTrain(relative_spikes * ms, 
                                                   t_stop=actual_time_window * ms, 
                                                   t_start=0 * ms)
                        # 计算instantaneous rate
                        inst_rate = instantaneous_rate(spiketrain, kernel=gk, sampling_period=bin_size_ms * ms)
                        response = inst_rate.magnitude.flatten()
                        
                        # 调整到标准长度
                        if actual_time_window < total_time_index:
                            # 如果实际时间窗口小于期望的，填充
                            if len(response) < n_time_bins_index:
                                response = np.pad(response, (0, n_time_bins_index - len(response)), 'constant')
                            elif len(response) > n_time_bins_index:
                                response = response[:n_time_bins_index]
                        elif len(response) > n_time_bins_index:
                            response = response[:n_time_bins_index]
                        elif len(response) < n_time_bins_index:
                            response = np.pad(response, (0, n_time_bins_index - len(response)), 'constant')
                    except (ValueError, IndexError) as e:
                        # 如果计算失败，使用零数组
                        response = np.zeros(n_time_bins_index)
                else:
                    response = np.zeros(n_time_bins_index)
            else:
                response = np.zeros(n_time_bins_index)
            
            # 确保长度一致
            if len(response) != n_time_bins_index:
                if len(response) < n_time_bins_index:
                    response = np.pad(response, (0, n_time_bins_index - len(response)), 'constant')
                else:
                    response = response[:n_time_bins_index]
            
            # 存储到反应矩阵
            response_matrix[neuron_idx, image_idx, trial_idx, :] = response

print(f"Response matrix shape: {response_matrix.shape}")
print(f"  - Neurons: {n_neurons}")
print(f"  - Images: {n_images}")
print(f"  - Max trials: {max_trials}")
print(f"  - Time bins: {n_time_bins_index}")

# 保存反应矩阵
response_matrix_path = os.path.join(output_dir, "response_matrix.npy")
np.save(response_matrix_path, response_matrix)
print(f"Saved response matrix to {response_matrix_path}")



Computing response matrix...
Matrix dimensions:
  - Neurons: 53
  - Images: 118
  - Max trials per image: 20
  - Time bins: 25
Response matrix shape: (53, 118, 20, 25)
  - Neurons: 53
  - Images: 118
  - Max trials: 20
  - Time bins: 25
Saved response matrix to /media/ubuntu/sda/mouse_test/sorted/WLF_128chmouse1_natima_RHD_251129_183351/response_matrix.npy


In [19]:
# 6. 训练和分类模型
# 导入PyTorch相关库
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import math
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

print("PyTorch libraries imported successfully")


PyTorch libraries imported successfully


In [20]:
# 定义Dataset类
class EPDataset(Dataset):
    def __init__(self, EP_data, labels, features):  
        self.EP_data = EP_data
        self.labels = labels
        self.features = features

    def __len__(self):
        return len(self.EP_data)
        
    def __getitem__(self, idx):
        EP_tensor = torch.tensor(self.EP_data[idx].T, dtype=torch.float32)  # 转置为 (time_bins, n_neurons)
        label = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        feature = self.features[idx]
        
        return EP_tensor, label, feature


In [21]:
# 模型配置类
class ModelConfig:
    def __init__(self,
                 input_neuron=41,        
                 time_bins=25,          
                 d_model = 150,          
                 nhead=10,                
                 num_transformer_layers=1, 
                 conv_channels=64,      
                 num_conv_blocks=3,      
                 num_classes=118,        
                 residual_dims=[256, 512, 1024], 
                 use_positional_encoding=True,  
                 dim_feedforward_ratio=4,      
                 activation='relu',
                 lr = 2e-4,
                 epochs = 200):
        
        # Transformer 
        self.transformer = {
            'd_model': d_model,
            'nhead': nhead,
            'num_layers': num_transformer_layers,
            'dim_feedforward': d_model * dim_feedforward_ratio,
            'activation': activation
        }
        
        # cnn
        self.convolution = {
            'channels': conv_channels,
            'num_blocks': num_conv_blocks,
            'kernel_size': (3, 3),
            'pool_size': (2, 2)
        }
        
        # resnet
        self.residual = {
            'dims': residual_dims,
            'skip_connection': True
        }

        self.input_dim = input_neuron
        self.time_steps = time_bins
        self.num_classes = num_classes
        self.positional_encoding = use_positional_encoding
        self.lr = lr
        self.epochs = epochs


In [22]:
# 位置编码类
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1)]
        return x

# Residual Linear Block
class ResidualLinearBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.activation = nn.GELU()
        self.downsample = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()

    def forward(self, x):
        residual = self.downsample(x)
        x = self.linear(x)
        x = self.norm(x)
        x = self.activation(x)
        return x + residual

# 主模型类
class TimeTransformerConvModel(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        self.input_proj = nn.Linear(config.input_dim, config.transformer['d_model'])
        self.pos_encoder = PositionalEncoding(config.transformer['d_model']) if config.positional_encoding else nn.Identity()
        
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=config.transformer['d_model'],
            nhead=config.transformer['nhead'],
            dim_feedforward=config.transformer['dim_feedforward'],
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(transformer_layer, config.transformer['num_layers'])
        
        self.conv_blocks = nn.Sequential()
        in_channels = 1
        for _ in range(config.convolution['num_blocks']):
            self.conv_blocks.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, config.convolution['channels'], 
                            kernel_size=config.convolution['kernel_size'], padding='same'),
                    nn.BatchNorm2d(config.convolution['channels']),
                    nn.ELU(),
                    nn.MaxPool2d(kernel_size=config.convolution['pool_size'])
                )
            )
            in_channels = config.convolution['channels']
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(config.convolution['channels'], config.num_classes)
        
        self.residual_layers = nn.Sequential()
        current_dim = config.convolution['channels']
        for dim in config.residual['dims']:
            self.residual_layers.append(ResidualLinearBlock(current_dim, dim))
            current_dim = dim
        if current_dim != 1024:
            self.residual_layers.append(nn.Linear(current_dim, 1024))
            self.residual_layers.append(nn.LayerNorm(1024))

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        
        x = x.unsqueeze(1)
        x = self.conv_blocks(x)
        x = self.adaptive_pool(x)
        x = x.flatten(1)
        
        logits = self.classifier(x)
        features = self.residual_layers(x)
        
        return logits, features


In [23]:
# 分类损失函数（简化版，只使用分类损失）
class ClassificationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, logits, labels):
        return self.ce_loss(logits, labels)


In [24]:
# 训练函数
def train_model(model, dataloader, optimizer, device, criterion, config):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (neuro, labels, _) in enumerate(dataloader):  # 忽略feature
        neuro = neuro.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        logits, _ = model(neuro)  # 忽略features输出
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            parameters=model.parameters(), 
            max_norm=3.0,                   
            norm_type=2.0                   
        )
        optimizer.step()
        
        # 统计指标
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
    train_loss = total_loss / len(dataloader)
    train_accuracy = correct / total
    return train_loss, train_accuracy

# 评估函数
@torch.no_grad()
def evaluate_model(model, dataloader, device, criterion, config):
    model.eval()
    total_loss = 0.0
    correct_top1 = 0
    correct_top5 = 0
    total = 0
    
    for neuro, labels, _ in dataloader:  # 忽略feature
        neuro = neuro.to(device)
        labels = labels.to(device)
        
        logits, _ = model(neuro)  # 忽略features输出
        
        loss = criterion(logits, labels)
        total_loss += loss.item()
        
        # Top-1 和 Top-5 准确率
        _, predicted_top1 = torch.max(logits, 1)
        correct_top1 += (predicted_top1 == labels).sum().item()
        _, predicted_top5 = logits.topk(5, dim=1)
        correct_top5 += torch.sum(predicted_top5.eq(labels.view(-1, 1))).item()
        
        total += labels.size(0)
    
    test_loss = total_loss / len(dataloader)
    test_accuracy = correct_top1 / total
    top5_accuracy = correct_top5 / total
    
    return test_loss, test_accuracy, top5_accuracy


In [25]:
# 主训练循环
def main_train_loop(config, model, train_loader, test_loader, device, early_stopping_patience=5, save_path=None):
    optimizer = AdamW(model.parameters(), lr=config.lr)
    criterion = ClassificationLoss()
    
    train_losses, train_accs = [], []
    test_losses, test_accs, test_top5 = [], [], []
    best_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    
    for epoch in range(config.epochs):
        train_loss, train_acc = train_model(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
            config=config
        )
        
        test_loss, test_acc, top5_acc = evaluate_model(
            model=model,
            dataloader=test_loader,
            device=device,
            criterion=criterion,
            config=config
        )
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        test_top5.append(top5_acc)
        
        # 早停机制
        if test_acc > best_acc:
            best_acc = test_acc
            best_epoch = epoch
            patience_counter = 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            
        # if patience_counter >= early_stopping_patience:
        #     print(f"Early stopping at epoch {epoch+1}, best accuracy: {best_acc:.2%}")
        #     break
        
        # 打印日志
        if epoch % 1 == 0 or epoch == config.epochs - 1:
            print(f"Epoch {epoch+1}/{config.epochs}")
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%}")
            print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%} | Top-5 Acc: {top5_acc:.2%}")
            print(f"Best Acc: {best_acc:.2%} (Epoch {best_epoch+1}) | Patience: {patience_counter}/{early_stopping_patience}")
            print("-" * 60)
    
    # 加载最佳模型
    if save_path and os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path))
    
    return {
        "best_test_acc": best_acc,
        "final_top5_acc": test_top5[-1],
        "train_history": {
            "loss": train_losses,
            "accuracy": train_accs
        },
        "test_history": {
            "loss": test_losses,
            "accuracy": test_accs,
            "top5_accuracy": test_top5
        },
        "best_epoch": best_epoch,
        "total_epochs": len(train_losses)
    }


In [26]:
# 准备训练数据：从response_matrix中提取每个trial的数据
print("Preparing training data from response_matrix...")

# 加载response_matrix
response_matrix_path = os.path.join(output_dir, "response_matrix.npy")
response_matrix = np.load(response_matrix_path)
print(f"Loaded response_matrix shape: {response_matrix.shape}")

# 验证unique_images的顺序是否一致（确保label正确）
print(f"\nVerifying unique_images order:")
print(f"First 10 images: {unique_images[:10]}")
print(f"Total images: {len(unique_images)}")

# 提取所有有效的trials（跳过NaN值）
EP_data = []  # 每个元素是 (n_neurons, time_bins) 的数组
labels = []   # 每个元素是image的索引（0-based）
image_to_label_map = {}  # 用于验证：image -> label

for image_idx, image in enumerate(unique_images):
    # 验证：确保image_idx与unique_images中的位置一致
    assert unique_images[image_idx] == image, f"Mismatch at index {image_idx}: expected {unique_images[image_idx]}, got {image}"
    image_to_label_map[image] = image_idx
    
    image_trials = trigger_log[trigger_log['image'] == image].reset_index(drop=True)
    
    for trial_idx in range(len(image_trials)):
        # 提取该trial的响应数据 (n_neurons, time_bins)
        trial_data = response_matrix[:, image_idx, trial_idx, :]
        
        # 检查是否有NaN值
        if not np.isnan(trial_data).any():
            EP_data.append(trial_data)  # (n_neurons, time_bins)
            labels.append(image_idx)  # 0-based索引

print(f"\nTotal valid trials: {len(EP_data)}")
print(f"Data shape per trial: {EP_data[0].shape}")
print(f"Number of unique labels: {len(set(labels))}")
print(f"Label range: {min(labels)} to {max(labels)}")

# 验证label是否正确对应到image
print(f"\nVerifying label mapping (first 5 images):")
for i in range(min(5, len(unique_images))):
    image = unique_images[i]
    label_count = labels.count(i)
    print(f"  Image '{image}' -> Label {i}: {label_count} trials")

# 创建虚拟特征（虽然不使用，但为了兼容Dataset接口）
features = [torch.randn(1024, dtype=torch.float32) for _ in range(len(labels))]

print("\nTraining data prepared successfully!")


Preparing training data from response_matrix...
Loaded response_matrix shape: (53, 118, 20, 25)

Verifying unique_images order:
First 10 images: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Total images: 118

Total valid trials: 2360
Data shape per trial: (53, 25)
Number of unique labels: 118
Label range: 0 to 117

Verifying label mapping (first 5 images):
  Image '0' -> Label 0: 20 trials
  Image '1' -> Label 1: 20 trials
  Image '2' -> Label 2: 20 trials
  Image '3' -> Label 3: 20 trials
  Image '4' -> Label 4: 20 trials

Training data prepared successfully!


In [27]:
# 创建训练和测试数据加载器
print("Creating data loaders...")

# 创建DataFrame用于分层采样
data_df = pd.DataFrame({
    'EP_data': EP_data,
    'labels': labels,
    'features': features
})

# 分层划分训练集和测试集
train_data, test_data = train_test_split(
    data_df, 
    test_size=0.2, 
    random_state=42,
    stratify=data_df['labels']
)

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

# 创建数据集
train_dataset = EPDataset(
    train_data['EP_data'].tolist(),
    train_data['labels'].tolist(),
    train_data['features'].tolist()
)

test_dataset = EPDataset(
    test_data['EP_data'].tolist(),
    test_data['labels'].tolist(),
    test_data['features'].tolist()
)

# 创建数据加载器
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Data loaders created successfully!")


Creating data loaders...
Train samples: 1888
Test samples: 472
Data loaders created successfully!


In [28]:
# 创建模型并开始训练
print("Creating model and starting training...")

# 模型配置
config = ModelConfig(
    input_neuron=n_neurons,      # 41
    time_bins=n_time_bins_index,  # 25
    d_model=150,
    nhead=10,
    num_transformer_layers=1,
    conv_channels=64,
    num_conv_blocks=3,
    num_classes=n_images,        # 118
    residual_dims=[256, 512, 1024],
    use_positional_encoding=True,
    dim_feedforward_ratio=4,
    activation='relu',
    lr=2e-4,
    epochs=200
)

# 创建模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = TimeTransformerConvModel(config).to(device)

# 保存路径
model_save_path = os.path.join(output_dir, "ep_encoder_model.pth")
result_save_path = os.path.join(output_dir, "ep_encoder_results.pkl")

# 开始训练
print("\n" + "="*60)
print("Starting training...")
print("="*60)

result = main_train_loop(
    config=config,
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    early_stopping_patience=5,
    save_path=model_save_path
)



print("\n" + "="*60)
print("Training completed!")
print("="*60)
print(f"Best Test Accuracy: {result['best_test_acc']:.2%}")
print(f"Final Top-5 Accuracy: {result['final_top5_acc']:.2%}")
print(f"Best Epoch: {result['best_epoch'] + 1}")
print(f"Model saved to: {model_save_path}")
print(f"Results saved to: {result_save_path}")


Creating model and starting training...
Using device: cuda

Starting training...
Epoch 1/200
Train Loss: 4.8321 | Train Acc: 0.79%
Test Loss: 4.7944 | Test Acc: 0.42% | Top-5 Acc: 4.45%
Best Acc: 0.42% (Epoch 1) | Patience: 0/5
------------------------------------------------------------
Epoch 2/200
Train Loss: 4.7836 | Train Acc: 0.85%
Test Loss: 4.7826 | Test Acc: 1.27% | Top-5 Acc: 4.24%
Best Acc: 1.27% (Epoch 2) | Patience: 0/5
------------------------------------------------------------
Epoch 3/200
Train Loss: 4.7760 | Train Acc: 1.06%
Test Loss: 4.7792 | Test Acc: 0.85% | Top-5 Acc: 3.60%
Best Acc: 1.27% (Epoch 2) | Patience: 1/5
------------------------------------------------------------
Epoch 4/200
Train Loss: 4.7654 | Train Acc: 1.17%
Test Loss: 4.7707 | Test Acc: 0.85% | Top-5 Acc: 5.08%
Best Acc: 1.27% (Epoch 2) | Patience: 2/5
------------------------------------------------------------
Epoch 5/200
Train Loss: 4.7309 | Train Acc: 2.01%
Test Loss: 4.7420 | Test Acc: 1.27% |